# Libraries and global variables

In [1]:
from pathlib import Path
import json
import sys
from typing import Optional
import time
#import config as cfg

import numpy as np
import pandas as pd
from PIL import Image

from fastai.vision.all import(
    DataBlock, ImageBlock, BBoxBlock, BBoxLblBlock,
    ColReader, ColSplitter,
    RandomSplitter, get_image_files,
    Resize, DataLoaders,
    cnn_learner, resnet50,
    SaveModelCallback, EarlyStoppingCallback, CSVLogger,
    accuracy, F1Score,
    show_results
)
# from fastai.data.transforms import parent_label

from fastai.vision.augment import (
    Brightness, Contrast, aug_transforms
)
from fastai.vision.core import imagenet_stats

import torch
from fastai.callback.fp16 import MixedPrecision

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
"/content/drive/MyDrive/Work/10 Foodbegood/01 original"

"/content/drive/MyDrive/Work/10 Foodbegood/01 original/lentils container big"
"/content/drive/MyDrive/Work/10 Foodbegood/01 original/lentils container small"
"/content/drive/MyDrive/Work/10 Foodbegood/01 original/rice container big"
"/content/drive/MyDrive/Work/10 Foodbegood/01 original/rice container small"

'/content/drive/MyDrive/Work/10 Foodbegood/01 original/rice container small'

In [5]:
!cp -r "/content/drive/MyDrive/Work/10 Foodbegood/01 model1" "/content/01 model1"

## Directory layout

In [ ]:
PLATFORM = "colab " # "colab", "kaggle", "local"

if PLATFORM == "colab":
  ROOT = Path("/content/01 model1")
elif PLATFORM == "kaggle":
  ROOT = Path("/kaggle/working/project_gn_food_estimator")
else:
  ROOT = Path(__file__).resolve().parent

In [ ]:
DATA_DIR      = ROOT / "data"
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EXPORTS_DIR   = DATA_DIR / "exports"
MODELS_DIR    = ROOT / "models"
TRAINING_DIR  = ROOT / "training"
LOGS_DIR      = TRAINING_DIR / "logs"

In [ ]:
# Labels file
COCO_MODEL1_JSON = EXPORTS_DIR / "model1_detection.json"

In [ ]:
# Model checkpoint path
MODEL1_PATH = MODELS_DIR / "model1_detector.pth"

In [ ]:
def setup_directories():
    """
    Ensure output directories exist.
    """
    MODELS_DIR.mkdir(parents = True, exist_ok = True)
    LOGS_DIR.mkdir(parents = True, exist_ok = True)
    print(f"[Setup] Models dir: {MODELS_DIR}")
    print(f"[Setup] Logs dir : {LOGS_DIR}")

In [ ]:
setup_directories()

## Parameters

In [ ]:
# Dataset split ratios
SPLIT_TRAIN = 0.80
SPLIT_VAL   = 0.10
SPLIT_TEST  = 0.10
RANDOM_SEED = 42

In [ ]:
# -----------------------------------------------------------------------------
# GN container standard specifications
# Two containers currently in scope: GN 1/1 (large) and GN 1/2 (small).
#
# Dimensions source: EN 631-1 standard
# Volume = inner_length × inner_width × depth (all mm → litres)
#
# Key:
#   gn_id       : string label used throughout the pipeline
#   label       : human-readable name (matches Model #1 class labels)
#   outer_l_mm  : outer length (mm)
#   outer_w_mm  : outer width (mm)
#   inner_l_mm  : inner length (mm)
#   inner_w_mm  : inner width (mm)
#   depth_mm    : internal depth (mm) — 65 mm default depth
#   volume_l    : nominal volume in litres
# -----------------------------------------------------------------------------
GN_CONTAINERS = {
    "GN_1_1": {
        "gn_id":      "GN_1_1",
        "label":      "large container",
        "outer_l_mm": 530,
        "outer_w_mm": 325,
        "inner_l_mm": 520,
        "inner_w_mm": 312,
        "depth_mm":   65,
        "volume_l":   round(520 * 312 * 65 / 1e6, 2),   # ≈ 10.55 L
    },
    "GN_1_2": {
        "gn_id":      "GN_1_2",
        "label":      "small container",
        "outer_l_mm": 325,
        "outer_w_mm": 265,
        "inner_l_mm": 312,
        "inner_w_mm": 254,
        "depth_mm":   65,
        "volume_l":   round(312 * 254 * 65 / 1e6, 2),   # ≈ 5.15 L
    },
}

In [ ]:
# Snap-to-standard tolerance: accept pixel-derived volume within this margin
GN_SNAP_TOLERANCE = 0.10   # 10% — snap to nearest GN if within this fraction

In [ ]:
# Model #1 class labels (must match Label Studio annotation labels exactly)
MODEL1_CLASSES = ["small container", "large container"]

## Training hyperparameters

In [ ]:
# Model #1 — ResNet-50, object detection head
MODEL1_BACKBONE        = "resnet50"
MODEL1_BATCH_SIZE      = 8
MODEL1_EPOCHS_FROZEN   = 5       # Phase 1: head only
MODEL1_EPOCHS_UNFROZEN = 10      # Phase 2: full fine-tune
MODEL1_LR_FROZEN       = 1e-3
MODEL1_LR_UNFROZEN     = 1e-4
MODEL1_MIXED_PREC      = True    # fp16

In [ ]:
def check_gpu():
    """
    Report GPU availability and VRAM.
    """
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

        print(f"[GPU] {gpu_name} - {vram_gb:.1f} GB VRAM")

        if vram_gb > 14.0:
            print(
                f"[GPU] WARNING: Less thatn 14 GB VRAM detected. "
                f"Consider reducing MODEL1_BATCH_SIZE."
                f"(currently {MODEL1_BATCH_SIZE})."
            )
    else:
        print("[GPU] No CUDA device found - training will run on CPU.")
    return "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
device = check_gpu()

# Data Preprocessing

In [ ]:
# Image pre-processing
IMAGE_SIZE = 512
NORM_MEAN = [0.485, 0.456, 0.406] # ImageNet statistics
NORM_STD = [0.229, 0.224, 0.225]
HIST_EQ_ENABLED = True

## Data augmentation pipeline

Augmentationt pipelines for Model #1. Each function returns a fastai Transform list that can be passed directly to a DataBlock or DataLoaders.

Augmentations applied (all toggleable):
- brightness
- rotation
- blur
- scaling
- flipping
- contrast

Design note: top-down food photography has specific characteristics:
- No meaningful vertical flip (gravity / food surface matters)
- Rotation up to ±30° is realistic (user hand angle variation)
- Moderate blur simulates slight camera shake at 25-28 cm distance
- Brightness / contrast variation simulates kitchen lighting changes

In [ ]:
# Data augmentation toggles / hyperparameters
AUG_BRIGHTNESS = True
AUG_ROTATION   = True
AUG_BLUR       = True
AUG_SCALING    = True
AUG_FLIPPING   = True
AUG_CONTRAST   = True

In [ ]:
def get_detection_transforms(size: int = IMAGE_SIZE):
    """
    Returns (train_tfms, valid_tfms) for Model #1 (object detection).

    For detection, geometric augmentations must be coordinate-aware (bounding
    boxes must transform with the image). Fastai's aug_transforms handles this
    automatically when used with a bounding-box DataBlock.

    Returns
        tuple[list, list]
            (train_transforms, valid_transforms)
    """
    aug_kwargs = dict(
        size = size,
        do_flip = AUG_FLIPPING,
        flip_vert = False,
        max_rotate = 15.0 if AUG_ROTATION else 0.0,
        min_zoom = 1.0,
        max_zoom = 1.1 if AUG_SCALING else 1.0,
        max_lighting = 0.25 if (AUG_BRIGHTNESS or AUG_CONTRAST) else 0.0,
        max_blur = 1.5 if AUG_BLUR else 0.0,
        max_warp = 0.0,
        p_affine = 0.75,
        p_lighting = 0.75
    )

    train_tfms = aug_transforms(**aug_kwargs)
    valid_tfms = []

    return train_tfms, vaild_tfms

In [ ]:
train_tfms, _ = get_detection_transforms(size = IMAGE_SIZE)

# Loading data

Dataset loader for:
- Model #1 — object detection (COCO JSON bounding-box annotations)

The function return a fastai DataLoaders object ready for training.

Label Studio COCO export format assumptions:
- Detection export  : standard COCO format with "images", "annotations", "categories" keys. Each annotation has bbox in [x_min, y_min, width, height] format.

Split strategy:
- Splits are performed at the image level (not annotation level) to avoid data leakage between train / val / test.
- Ratios: 80 / 10 / 10
- Random seed is fixed for reproducibility.

In [ ]:
# Shared helpers

def _load_coco_json(json_path: Path) -> dict:
    """
    Load and validate a COCO JSON file exported from Label Studio.
    """
    if not json_path.exists():
        raise FileNotFoundError(
            f"COCO annotation file not found: {json_path}\n"
            f"Export your Label Studio project as COCO JSON and place it at "
            f"the path defined in {COCO_MODEL1_JSON}."
        )
    with open(json_path, "r") as f:
        data = json.load(f)
    required_keys = {"images", "annotations", "categories"}
    if not required_keys.issubset(data.keys()):
        raise ValueError(
            f"COCO JSON at {json_path} is missing keys: "
            f"{required_keys - data.keys()}"
        )
    return data


def _split_image_ids(
        image_ids: list[int],
        train_ratio: float = SPLIT_TRAIN,
        val_ratio: float = SPLIT_VAL,
        seed: int = RANDOM_SEED,
    ) -> tuple[set[int], set[int], set[int]]:
    """
    Randomly split a list of image IDs into train / val / test sets.

    Returns:
        tuple[set, set, set]
            (train_ids, val_ids, test_ids)
    """
    rng = np.random.default_rng(seed)
    ids = np.array(image_ids)
    rng.shuffle(ids)

    n = len(ids)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_ids = set(ids[:n_train].tolist())
    val_ids = set(ids[n_train:n_train + n_val].tolist())
    test_ids = set(ids[n_train + n_val:].tolist())

    print(
        f"Split -> train: {len(train_ids)} | val: {len(val_ids)} | test: {len(test_ids)} images"
    )
    return train_ids, val_ids, test_ids

## Build DataLoaders

In [ ]:
def get_detection_dataloaders(
        coco_json_path: Path = COCO_MODEL1_JSON,
        image_dir: Path = RAW_DIR,
        image_size: int = IMAGE_SIZE,
        batch_size: int = MODEL1_BATCH_SIZE,
        train_tfms: list = None,
        device: str = "cuda",
        verbose: bool = True
    ) -> tuple[DataLoaders, pd.DataFrame, pd.DataFrame]:
    """
    Build fastai DataLoaders for bounding-box object detection (model #1).

    Parses a Label Studio COCO JSON export and constructs a DataFrame with one
    row per annotation, then hands it to a fastai DataBlock.

    Parameters:
        coco_json_path : Path
            Path to the COCO JSON annotation file.
        image_dir : Path
            Directory containing the raw images.
        image_size : int
            Target image size (longer edge).
        batch_size : int
            Training batch size.
        train_tfms : list, optional
            Fastai transforms for training set. If None, uses aug_transforms defaults.
        device : str
            "cuda" or "cpu"
        verbose : bool
            Print split and class information.

    Returns:
        tuple[DataLoaders, pd.DataFrame, pd.DataFrame]
            (dls, train_df, test_df) - test_df is returned separately for
            post-training evaluation; it is NOT included in the DataLoaders.
    """
    if verbose:
        print(f"Loading COCO annonations from: {coco_json_path}")

    coco = _load_coco_json(coco_json_path)

    # Build category ID -> label name mapping
    id_to_label = {cat["id"]: cat["name"] for cat in coco["categories"]}
    if verbose:
        print(f"Categories: {id_to_label}")

        # Warn if labels do not match config
        coco_labels = set(id_to_label.values())
        cfg_labels = set(MODEL1_CLASSES)
        if coco_labels != cfg_labels:
            print(
                f"WARNING: COCO labels {coco_labels} differ from "
                f"configuration {cfg_labels}."
                f"Update MODEL1_CLASSES if needed."
            )

    # Build image ID -> filename mapping
    id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
    all_images_ids = list(id_to_file.keys())

    # Split image IDs
    if verbose:
        print(f"Total images: {len(all_image_ids)}")
    train_ids, val_ids, test_ids = _split_image_ids(all_image_ids)

    # Build annotation DataFrame
    # COCO bbox: [x_min, y_min, width, height] -> fastai want [x_min, y_min, x_max, y_max]
    rows = []
    for ann in coco["annotations"]:
        img_id = ann["image_id"]
        filename = id_to_file[img_id]
        x, y, w, h = ann["bbox"]
        label = id_to_label[ann["category_id"]]
        split = (
            "train" if img_id in train_ids else
            "valid" if img_id in val_ids else
            "test"
        )
        rows.append({
            "image_path": str(image_dir / filename),
            "x_min": x,
            "y_min": y,
            "x_max": x + w,
            "y_max": y + h,
            "label": label,
            "split": split
        })

    full_df = pd.DataFrame(rows)

    # Separate test set - not fed into DataLoaders
    test_df = full_df[full_df["split"] == "test"].copy()
    train_df = full_df[full_df["split"] != "test"].copy()
    train_df["is_valid"] = train_df["split"] == "valid"

    if verbose:
        print(f"Annotation rows -> train + val: {len(train_df)} | test: {len(test_df)}")


    # Aggregate annotations per image into lists (fastai bbox format)
    def _agg_bboxes(grp):
        bboxes = grp[["x_min", "y_min", "x_max", "y_max"]].values.tolist()
        labels = grp["label"].tolist()
        return pd.Series({
            "image_path": grp["image_path"].iloc[0],
            "bboxes": bbboxes,
            "labels": labels,
            "is_valid": grp["is_valid"].iloc[0]
        })


    agg_df = (
        train_df
        .groupby("image_path", group_keys = False)
        .apply(_agg_bboxes)
        .reset_index(drop=True)
    )

    # Build fastai DataBlock
    # Note: BBoxBlock + BBoxLblBlock require paired getters
    dblock = DataBlock(
        blocks = (ImageBlock, BBoxBlock, BBoxLblBlock(add_na = True)),
        n_inp = 1,
        get_x = ColReader("image_path")
        get_y = [ColReader("bboxes"), ColReader("labels")],
        splitter = ColSplitter("is_valid"),
        item_tfms = Resize(image_size),
        batch_tfms = train_tfms
    )

    dls = dblock.dataloaders(
        agg_df,
        bs = batch_size,
        devive = device
    )

    if verbose:
        print(f"DataLoaders ready - vocab: {dls.vocab}")

    return dls, train_df, test_df

In [ ]:
dls, train_df, test_df = get_detection_dataloaders(
    coco_json_path = COCO_MODEL1_JSON,
    image_dir = RAW_DIR,
    image_size = IMAGE_SIZE,
    batch_size = MODEL1_BATCH_SIZE,
    train_tfms = train_tfms,
    device = device,
    verbose = True
)

In [ ]:
# Save test split for later evaluation
test_csv_path = LOGS_DIR / "model1_test_split.csv"
test_df.to_csv(test_csv_path, index = False)
print("Test split saved: {test_csv_path}")

In [ ]:
# Peek at a batch
print("Sample batch:")
xb, *yb = dls.one_batch()
print(f" Image shape: {xb.shape}")
print(f" Number classes: {len(dls.vocab)}")
print(f" Vocab: {dls.vocab}")

# Constructing learner

Train Model #1: GN container object detection

**Architecture:**

ResNet-50 backbone (ImageNet pre-trained) with a bounding-box regression head via fastai's cnn_learner.

**Strategy: Two-phase training**

- Phase 1 — backbone frozen, head trained for MODEL1_EPOCHS_FROZEN

- Phase 2 — full network unfrozen, fine-tuned with discriminative learning rates for MODEL1_EPOCHS_UNFROZEN

Mixed precision: fp16 enabled (safe on T4, gives ~2× speedup)

In [ ]:
log_path = LOGS_DIR / "model1_training_log.csv"

In [ ]:
learn = cnn_learner(
    dls,
    resnet50,
    metrics = [accuracy], # swap for AP/mAP once a detection head is wired in
    model_dir = MODELS_DIR,
    cbs = [
        SaveModelCallback(
            monitor = "valid_loss",
            fname = "model1_best",
            with_opt = True     # save optimiser state for resuming
        ),
        EarlyStoppingCallback(
            monitor = "valid_loss",
            patience = 4    # stop if no improvement for 4 epochs
        ),
        CSVLogger(fname = str(log_path))
    ]
)

In [ ]:
# Enable mixed precision (fp16) for T4 speedup
if MODEL1_MIXED_PREC and device == "cuda":
    learn = learn.to_fp16()
    print("Mixed precision (fp16) - ENABLED")

# Phase 1 - frozen backbone, train head only

In [ ]:
learn.freeze()

## Find learning rate

In [ ]:
suggested_lr = MODEL1_LR_FROZEN
try:
    lr_finder = learn.lr_find(suggest_funcs = None, show_plot = False)
    suggested_lr = lr_finder.valley
    print(f"lr_find suggestion: {suggested_lr:.2e}"
        f"(config default: {MODEL1_LR_FROZEN:.2e})")
except Exception as e:
    print(f"lr_find failed ({e}), using config default: {suggested_lr:.2e}")

t0 = time.time()
learn.fit_one_cycle(
    MODEL1_EPOCHS_FROZEN,
    lr_max = suggested_lr,
)
elapsed = time.time() - t0
print("Phase 1 complete - {elapsed:.0f}")

# Phase 2 - unfreeze all layers, fine-tune with discriminative LRs

In [ ]:
learn.unfreeze()

## Discriminative learning rates
earlier layers get lower LR

slice (LR/100) is the standard fastai recommendation

In [ ]:
lr_unfrozen = MODEL1_LR_UNFROZEN
print(f"LR slice: {lr_unfrozeen / 100:.2e} -> {lr_unfrozen:.2e}")

t0 = time.time()
learn.fit_one_cycle(
    MODEL1_EPOCHS_UNFROZEN,
    lr_max = slice(lr_unfrozen / 100 , lr_unfrozen)
)
elapsed = time.time() - t0
print(f"Phase 2 complete - {elapsed:.0f}s")

# Export final model

In [ ]:
final_path = MODEL1_PATH
learn.export(final_path)
print(f"[Done] Model exported: {final_path}")

In [ ]:
# Also save the raw state dict for maximum portability
state_dict_path = MODELS_DIR / "model1_state_dict.pth"
torch.save(learn.model.state_dict(), state_dict_path)
print(f"[Done] State dict: {state_dict_path}")

# Evaluation on validation set

In [ ]:
val_results = learn.validate()

print(f"valid_loss: {val_reuslts[0]:.4f}")

for i, m in enumerate(learn.metrics):
    print(f"{m.name:15s}: {val_results[i + 1]:.4f}")

# Results

# Save trained model

# Load trained model

## Export model for production

# Predictions / Inference

## Save predictions